# SynthSara — Phone APK Builder

**One-cell Google Colab builder for the native SynthSara Android launcher.**

This notebook builds the Android launcher from the GitHub branch used by the v0.1 device-acceptance runbook. It does **not** root, unlock, flash, wipe, or modify Android system files.

### Phone instructions
1. Open this notebook in Google Colab.
2. Tap the play button on the large code cell below.
3. Keep the browser tab open while the Android SDK and Gradle install and the project builds.
4. If successful, `SynthSara-Android-v0.1-debug.apk` downloads automatically.
5. If compilation fails, `SynthSara-Phone-APK-build.log` downloads automatically. Send that log back for repair.

> Build source defaults to `chaosweaver007/synthsara-node-zero` → `feat/android-launcher-v0.1`.


In [ ]:
# SYNTHSARA PHONE APK BUILDER — run this cell
from google.colab import files
from pathlib import Path
import os, shutil, subprocess, sys, hashlib, textwrap, urllib.request, zipfile

REPO = "https://github.com/chaosweaver007/synthsara-node-zero.git"
REF = "feat/android-launcher-v0.1"
EXPECTED_HEAD = "3b258e7d0fec62eef7d91ebb1d62dd84e931832a"
WORK = Path("/content/synthsara_phone_build")
SRC = WORK / "repo"
SDK = Path("/content/android-sdk")
GRADLE_HOME = Path("/content/gradle-8.9")
LOG = Path("/content/SynthSara-Phone-APK-build.log")
OUT_APK = Path("/content/SynthSara-Android-v0.1-debug.apk")
log_lines = []

def emit(msg=""):
    print(msg, flush=True)
    log_lines.append(str(msg))

def run(cmd, cwd=None, env=None, allow_failure=False):
    emit("\n$ " + " ".join(map(str, cmd)))
    p = subprocess.Popen(
        list(map(str, cmd)),
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in p.stdout:
        print(line, end="")
        log_lines.append(line.rstrip("\n"))
    code = p.wait()
    if code and not allow_failure:
        raise subprocess.CalledProcessError(code, cmd)
    return code

def save_log():
    LOG.write_text("\n".join(log_lines) + "\n", encoding="utf-8")

try:
    emit("🔥 SynthSara Android phone build starting")
    emit(f"Repository: {REPO}")
    emit(f"Ref: {REF}")

    if WORK.exists():
        shutil.rmtree(WORK)
    WORK.mkdir(parents=True)

    # JDK 17
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk", "unzip", "wget", "git"])

    java_home = Path("/usr/lib/jvm/java-17-openjdk-amd64")
    env = os.environ.copy()
    env["JAVA_HOME"] = str(java_home)
    env["PATH"] = f"{java_home / 'bin'}:{env['PATH']}"

    # Android command-line tools + SDK 35
    SDK.mkdir(parents=True, exist_ok=True)
    cmdline_zip = Path("/content/commandlinetools-linux.zip")
    cmdline_root = SDK / "cmdline-tools"
    latest = cmdline_root / "latest"
    if not (latest / "bin" / "sdkmanager").exists():
        emit("Installing Android command-line tools...")
        urllib.request.urlretrieve(
            "https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip",
            cmdline_zip
        )
        temp = Path("/content/android-cmdline-temp")
        if temp.exists():
            shutil.rmtree(temp)
        temp.mkdir()
        with zipfile.ZipFile(cmdline_zip) as z:
            z.extractall(temp)
        latest.parent.mkdir(parents=True, exist_ok=True)
        if latest.exists():
            shutil.rmtree(latest)
        shutil.move(str(temp / "cmdline-tools"), str(latest))

    env["ANDROID_HOME"] = str(SDK)
    env["ANDROID_SDK_ROOT"] = str(SDK)
    env["PATH"] = f"{latest / 'bin'}:{SDK / 'platform-tools'}:{env['PATH']}"

    yes = subprocess.Popen(
        ["bash", "-lc", f"yes | {latest / 'bin' / 'sdkmanager'} --licenses >/dev/null"],
        env=env
    )
    if yes.wait() != 0:
        raise RuntimeError("Android SDK license acceptance failed.")

    run([
        latest / "bin" / "sdkmanager",
        "platform-tools",
        "platforms;android-35",
        "build-tools;35.0.0",
    ], env=env)

    # Gradle 8.9 — mirrors the green GitHub Actions workflow.
    if not (GRADLE_HOME / "bin" / "gradle").exists():
        emit("Installing Gradle 8.9...")
        gradle_zip = Path("/content/gradle-8.9-bin.zip")
        urllib.request.urlretrieve(
            "https://services.gradle.org/distributions/gradle-8.9-bin.zip",
            gradle_zip
        )
        with zipfile.ZipFile(gradle_zip) as z:
            z.extractall("/content")

    env["PATH"] = f"{GRADLE_HOME / 'bin'}:{env['PATH']}"

    # Source
    run(["git", "clone", "--depth", "1", "--branch", REF, REPO, SRC], env=env)
    actual_head = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=SRC, env=env, text=True
    ).strip()
    emit(f"Checked out: {actual_head}")
    if actual_head != EXPECTED_HEAD:
        emit(
            "NOTE: branch head differs from the original v0.1 tested commit. "
            "Building current branch head rather than stopping."
        )

    android = SRC / "android"
    (android / "local.properties").write_text(
        f"sdk.dir={SDK}\n", encoding="utf-8"
    )

    # Same verification/build sequence as the green GitHub Actions workflow.
    run([sys.executable, "scripts/verify_privacy_contract.py"], cwd=android, env=env)
    run([
        GRADLE_HOME / "bin" / "gradle",
        ":app:lintDebug",
        ":app:assembleDebug",
        "--no-daemon",
        "--stacktrace",
    ], cwd=android, env=env)

    built_apk = android / "app" / "build" / "outputs" / "apk" / "debug" / "app-debug.apk"
    if not built_apk.exists():
        raise FileNotFoundError(f"Build completed but APK was not found: {built_apk}")

    shutil.copy2(built_apk, OUT_APK)
    digest = hashlib.sha256(OUT_APK.read_bytes()).hexdigest()
    emit(f"\n✅ BUILD GREEN")
    emit(f"APK: {OUT_APK.name}")
    emit(f"SHA-256: {digest}")
    emit("Downloading APK to your phone...")
    save_log()
    files.download(str(OUT_APK))

except Exception as exc:
    emit(f"\n❌ BUILD FAILED: {type(exc).__name__}: {exc}")
    save_log()
    print(f"\nBuild log saved to {LOG}")
    files.download(str(LOG))
    raise
